In [21]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

Create a Second Version of the Dataset (Cleaned Data)

In [22]:

df = pd.read_csv("C:\\MachineLearningRating_v3.txt", sep="|", encoding="utf-8")

# Example cleaning operations
df = df.drop_duplicates()

df["TransactionMonth"] = pd.to_datetime(df["TransactionMonth"])

# Remove negative claims
df = df[df["TotalClaims"] >= 0]

# Fill missing custom values
df["CustomValueEstimate"] = (
    df["CustomValueEstimate"]
      .fillna(df["CustomValueEstimate"].median())
)

df.to_csv("C:\\insurance-risk-analytics\\data\\MachineLearningRating_v3_cleaned.txt", index=False)

C:\Users\bewha\AppData\Local\Temp\ipykernel_21200\2355169702.py:1: DtypeWarning: Columns (0: CapitalOutstanding, 1: CrossBorder) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("C:\\MachineLearningRating_v3.txt", sep="|", encoding="utf-8")


In [23]:
df["HasClaim"] = (df["TotalClaims"] > 0).astype(int)

df["ClaimSeverity"] = np.where(
    df["TotalClaims"] > 0,
    df["TotalClaims"],
    np.nan
)

df["Margin"] = df["TotalPremium"] - df["TotalClaims"]

In [24]:
print(df.columns.tolist())

['UnderwrittenCoverID', 'PolicyID', 'TransactionMonth', 'IsVATRegistered', 'Citizenship', 'LegalType', 'Title', 'Language', 'Bank', 'AccountType', 'MaritalStatus', 'Gender', 'Country', 'Province', 'PostalCode', 'MainCrestaZone', 'SubCrestaZone', 'ItemType', 'mmcode', 'VehicleType', 'RegistrationYear', 'make', 'Model', 'Cylinders', 'cubiccapacity', 'kilowatts', 'bodytype', 'NumberOfDoors', 'VehicleIntroDate', 'CustomValueEstimate', 'AlarmImmobiliser', 'TrackingDevice', 'CapitalOutstanding', 'NewVehicle', 'WrittenOff', 'Rebuilt', 'Converted', 'CrossBorder', 'NumberOfVehiclesInFleet', 'SumInsured', 'TermFrequency', 'CalculatedPremiumPerTerm', 'ExcessSelected', 'CoverCategory', 'CoverType', 'CoverGroup', 'Section', 'Product', 'StatutoryClass', 'StatutoryRiskType', 'TotalPremium', 'TotalClaims', 'HasClaim', 'ClaimSeverity', 'Margin']


## Hypothesis-by-Hypothesis Setup
## 1. Province Risk Difference

In [38]:
from scipy.stats import chi2_contingency
import pandas as pd

# Contingency table for ALL provinces
table = pd.crosstab(df["Province"], df["HasClaim"])

# Chi-square test
chi2, p, dof, expected = chi2_contingency(table)

print("Chi-square statistic:", chi2)
print("P-value:", p)
print("Degrees of freedom:", dof)

if p < 0.05:
    print("Reject H0: Risk differs across provinces")
else:
    print("Fail to Reject H0: No significant provincial risk difference")

Chi-square statistic: 104.19041870171219
P-value: 5.926803398618002e-19
Degrees of freedom: 8
Reject H0: Risk differs across provinces


## 2.Margin Difference Between Zip Codes

In [39]:
top_zipcodes = df["PostalCode"].value_counts().head(2).index.tolist()

df_zip = df[df["PostalCode"].isin(top_zipcodes)]

table = pd.crosstab(df_zip["PostalCode"], df_zip["HasClaim"])

chi2, p, dof, expected = chi2_contingency(table)

print("Chi-square statistic:", chi2)
print("P-value:", p)
print("Degrees of freedom:", dof)

if p < 0.05:
    print("Reject H0")
else:
    print("Fail to Reject H0")

Chi-square statistic: 3.5970507124283615
P-value: 0.05788217295685014
Degrees of freedom: 1
Fail to Reject H0


In [40]:
df["Margin"] = df["TotalPremium"] - df["TotalClaims"]

In [41]:
from scipy.stats import ttest_ind

group_a = df_zip[df_zip["PostalCode"] == top_zipcodes[0]]["Margin"]

group_b = df_zip[df_zip["PostalCode"] == top_zipcodes[1]]["Margin"]

t_stat, p = ttest_ind(
    group_a,
    group_b,
    equal_var=False,
    nan_policy="omit"
)

print("P-value:", p)

if p < 0.05:
    print("Reject H0")
else:
    print("Fail to Reject H0")

P-value: 0.24446241842452013
Fail to Reject H0


## Gender Risk Difference

In [42]:
table = pd.crosstab(df["Gender"], df["HasClaim"])

chi2, p, dof, expected = chi2_contingency(table)

print("P-value:", p)

if p < 0.05:
    print("Reject H0")
else:
    print("Fail to Reject H0")

P-value: 0.02656634481035202
Reject H0
